# Chapter 6 Walkthrough: HCP and Account Targeting

This executed notebook builds the Chapter 6 artifacts at the December 31, 2024 cutoff. Run `ch06_hcp/scripts/generate_ch06_data.py` before rebuilding the notebook.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "ch06_hcp").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook inside the repository.")
    ROOT = ROOT.parent

SCRIPT_DIR = ROOT / "ch06_hcp" / "scripts"
sys.path.insert(0, str(SCRIPT_DIR))

analysis_module = importlib.import_module("run_analysis")
targeting = importlib.import_module("targeting")
referral = importlib.import_module("referral_network")
segmentation = importlib.import_module("segmentation")

results = analysis_module.run_analysis(ROOT)

headline = pd.Series({
    "Journey patients": results["attribution_comparison"].patient_id.nunique(),
    "Eligible-roster patients": results["patient_hcp"].patient_id.nunique(),
    "Eligible HCPs": results["hcp_features"].npi.nunique(),
    "Eligible accounts": results["account_targets"].account_id.nunique(),
})
print(headline.to_frame("count"))


                          count
Journey patients           6393
Eligible-roster patients   1556
Eligible HCPs               158
Eligible accounts           114


The eligible roster covers 1,556 patients and 158 HCPs across 114 site accounts. The remaining journey patients stay outside this field-planning artifact.


## 1. Attribution sensitivity


In [2]:
agreement = results["attribution_summary"].copy()
agreement["agreement_rate"] = agreement["agreement_rate"].map(
    lambda value: f"{value:.1%}"
)
print(agreement)
print(
    results["attribution_comparison"].query(
        "patient_id == 'PAT02034'"
    )
)


           comparison  patients_with_both  same_hcp agreement_rate
0  Index vs plurality                6393      4399          68.8%
1     Index vs latest                6393      4088          63.9%
2         All 3 rules                6393      4005          62.6%
    patient_id   index_npi plurality_npi  latest_npi  all_rules_agree
635   PAT02034  9000000280    9000000280  9000000280             True


All 3 attribution rules agree for 63.4% of patients. PAT02034 remains assigned to HCP0280 under every rule.


## 2. HCP evidence and concentration


In [3]:
columns = [
    "npi", "account_id", "cohort_patients", "treated_patients",
    "roventra_starts", "competitor_treated", "untreated_mature",
    "review_opportunity", "contact_permission_status",
]
print(
    results["hcp_features"].sort_values(
        ["cohort_patients", "npi"], ascending=[False, True]
    )[columns].head(10)
)
print(results["decile_summary"].head())


            npi account_id  cohort_patients  treated_patients  \
93   9000000430     ACC189               36                 9   
105  9000000469     ACC121               34                10   
35   9000000162     ACC062               33                13   
99   9000000447     ACC216               32                12   
5    9000000026     ACC226               28                11   
121  9000000537     ACC079               28                 6   
49   9000000217     ACC224               27                 8   
117  9000000516     ACC167               27                 9   
102  9000000460     ACC219               26                10   
86   9000000389     ACC155               24                 8   

     roventra_starts  competitor_treated  untreated_mature  \
93                 2                   7                25   
105                9                   1                21   
35                 8                   5                19   
99                 5                

The highest-volume rows include opt-outs. The first 30% of HCPs capture 55.2% of review opportunity.


![Top 20 HCPs ranked by review opportunity. Blue bars show review opportunity for Allowed HCPs, red bars for Opt-out HCPs, gray for Unknown. Light blue shows remaining attributed patients.](assets/figures/figure_6_1_volume_diagnostic.svg)


![Line chart starting at the origin showing cumulative review opportunity captured as contactable HCP share increases, ranked by review opportunity, with a dashed diagonal reference line.](assets/figures/figure_6_2_cumulative_capture.svg)


## 3. Referral pathways


In [4]:
print(results["referral_edges"].head(10))
stable = results["referral_metrics"].merge(
    results["referral_stability"][[
        "npi", "bootstrap_top20_frequency", "window_rank_range",
    ]],
    on="npi",
)
print(stable.head(15))


   source_npi destination_npi source_specialty destination_specialty  \
0  9000000578      9000000258     Primary Care         Endocrinology   
1  9000000417      9000000164     Primary Care         Endocrinology   
2  9000000460      9000000567     Primary Care         Endocrinology   
3  9000000033      9000000302     Primary Care         Endocrinology   
4  9000000265      9000000409     Primary Care         Endocrinology   
5  9000000520      9000000127     Primary Care         Endocrinology   
6  9000000020      9000000409     Primary Care         Endocrinology   
7  9000000128      9000000567     Primary Care         Endocrinology   
8  9000000470      9000000217     Primary Care         Endocrinology   
9  9000000565      9000000217     Primary Care         Endocrinology   

  source_account_id destination_account_id   region  unique_patients  \
0            ACC142                 ACC164  Midwest               22   
1            ACC126                 ACC109     West            

The referral output is a pathway-context artifact. Stability comes from transition-window comparison and patient-level bootstrap resampling.


![Schematic referral graph illustrating directed edges, patient counts, and betweenness centrality.](assets/figures/figure_6_3_referral_schematic.svg)


![Directed account-centered referral network with patient counts on each edge.](assets/figures/figure_6_4_referral_network.svg)


## 4. Scientific role evidence


In [5]:
candidates = results["kol_profiles"].loc[
    results["kol_profiles"]["kol_candidate"]
]
print(candidates[[
    "npi", "specialty_1", "research_percentile",
    "leadership_percentile", "practice_expertise_percentile",
    "peer_connection_percentile", "proposed_role",
    "role_fit_score", "evidence_confidence",
]].head(15))
print(results["kol_validation"])
print(results["kol_transparency_review"].head())


           npi    specialty_1  research_percentile  leadership_percentile  \
0   9000000105     Cardiology           100.000000              80.000000   
1   9000000206  Endocrinology           100.000000               8.333333   
2   9000000211  Endocrinology           100.000000              36.363636   
3   9000000237   Primary Care           100.000000              75.000000   
4   9000000363  Endocrinology           100.000000             100.000000   
5   9000000441   Primary Care           100.000000              84.615385   
6   9000000512     Cardiology           100.000000              31.250000   
7   9000000562     Cardiology           100.000000              40.000000   
8   9000000633   Primary Care           100.000000              92.592593   
9   9000000366  Endocrinology            96.153846               4.166667   
10  9000000277     Cardiology            94.736842              25.000000   
11  9000000258  Endocrinology            92.307692              41.666667   

Scientific role fit and payment transparency remain separate. Medical affairs owns the role review.


![Four scatter plots, one per proposed scientific role, showing candidate positions on that role's two primary evidence dimensions. Gray dots show candidates from other roles for context.](assets/figures/figure_6_5_kol_evidence_matrix.svg)


## 5. K-means engagement archetypes


In [6]:
print(results["cluster_evaluation"])
print(results["segment_profiles"])
print(results["segment_policy_comparison"])


   k    inertia  silhouette  minimum_cluster_size  minimum_cluster_share  \
0  3  55.664499    0.295087                    15               0.267857   
1  4  43.864938    0.291884                    10               0.178571   
2  5  34.477920    0.318859                     7               0.125000   
3  6  29.600367    0.298179                     7               0.125000   

   seed_stability_ari  bootstrap_stability_ari  selection_score  \
0            1.000000                 0.594227         0.793644   
1            1.000000                 0.555224         0.780690   
2            1.000000                 0.713949         0.830679   
3            0.746004                 0.603407         0.718865   

   operational_size_pass  selected  
0                   True      True  
1                   True     False  
2                  False     False  
3                  False     False  
   cluster_id  hcp_count  cohort_patients  opportunity_rate  roventra_share  \
0           0      

The selected 4-cluster solution has silhouette 0.427, seed ARI 1.000, bootstrap ARI 0.858, and minimum cluster size 10.


![2×2 small-multiples bar charts showing each engagement archetype's evidence-need, access-need, digital-response, and field-response scores.](assets/figures/figure_6_6_segment_profiles.svg)


![Line chart comparing silhouette, seed ARI, and bootstrap ARI for candidate cluster counts.](assets/figures/figure_6_7_cluster_validation.svg)


## 6. Account policy and call plan


In [7]:
print(results["gate_summary"])
print(
    results["account_targets"].set_index("account_id").loc[
        ["ACC155", "ACC002", "ACC121", "ACC231"],
        ["account_action", "reason_code", "cohort_patients",
         "treated_patients", "review_opportunity", "roventra_share"],
    ]
)
print(results["call_plan"])


                    stage  accounts
0  Eligible site accounts       114
1          Evidence floor        70
2     Treated denominator        20
3       Opportunity floor        20
4  No access-review route        20
5        Field permission        15
6  Ownership and capacity        15
               account_action                        reason_code  \
account_id                                                         
ACC155      Increase priority      PRIORITIZE_REVIEW_OPPORTUNITY   
ACC002          Access review                ROUTE_ACCESS_REVIEW   
ACC121           Hold contact                 HOLD_NO_PERMISSION   
ACC231                Monitor  MONITOR_SMALL_TREATED_DENOMINATOR   

            cohort_patients  treated_patients  review_opportunity  \
account_id                                                          
ACC155                   38                15                  31   
ACC002                   14                 6                  11   
ACC121                   34

The policy produces 9 priority accounts and a 19-call plan across 10 permitted HCPs. The output retains non-executable monitor, access-review, and hold queues.


![Scatter plot with priority accounts shown as large labeled dots in the foreground and non-priority accounts as small faded dots, with a shaded priority zone.](assets/figures/figure_6_8_account_actions.svg)


![Horizontal gate chart showing accounts remaining after evidence, treated denominator, opportunity, access, permission, ownership, and capacity gates.](assets/figures/figure_6_9_gate_attrition.svg)


## 7. Territory and policy review


In [8]:
print(results["plan_comparison"])
print(results["territory_summary"])
print(
    results["policy_sensitivity"].query(
        "minimum_account_patients == 10"
    )
)


                       plan  selected_hcps  contact_permitted  \
0  Top 30 by patient volume             30                 30   
1   Gated 4-week field plan             11                 11   

   held_or_unknown  review_opportunity  recent_contacts  
0                0                 397               43  
1                0                 143                6  
  territory  accounts  priority_accounts  review_opportunity  \
0       T06        17                  3                 184   
1       T03        10                  1                 120   
2       T04         9                  1                  88   
3       T01        16                  2                 178   
4       T08        18                  0                 196   
5       T07        12                  1                 133   
6       T05        16                  1                 148   
7       T02        16                  0                 137   

   actionable_opportunity  planned_hcps  recommended_

T03 has actionable opportunity and no executable calls. The gap belongs in field-leadership review and the override record.


![Dumbbell chart comparing each territory's actionable opportunity share (blue) and planned call share (green), with red connecting lines flagging zero-call territories.](assets/figures/figure_6_10_territory_allocation.svg)


## 8. Export the evidence package


In [9]:
output_dir = ROOT / "ch06_hcp" / "assets" / "generated_outputs"
analysis_module.write_outputs(results, output_dir, ROOT)
print(f"Wrote {len(results)} CSV artifacts and manifest.json")


Wrote 32 CSV artifacts and manifest.json


The package carries analysis date, source hashes, rule version, decision boundaries, and output contracts.
